# Inspect a saved result

Reads one baseline result (`SCENARIOS_DIR/results_<N>GB/<query>.parquet`, saved by `save_baseline.py`) and shows it as text. No Spark needed: pyarrow reads the Parquet folder directly.

In [ ]:
import sys

import pyarrow.parquet as pq

sys.path.insert(0, "../pipeline")
from constants import get_results_dir
from toolkit.results import results_path

SCALE_FACTOR = 1
QUERY_FILE = "query72.sql"
STATEMENT_PART = None  # only for files with several statements (14, 23, 24, 39)

path = results_path(get_results_dir(SCALE_FACTOR), QUERY_FILE, STATEMENT_PART)
table = pq.read_table(path)
print(path)
print(table.schema)

## As a string

One line per row, tab-separated, header first. NULL is written as `NULL` so it can't be confused with an empty string, and values use their exact Python form (`Decimal('1.50')` → `1.50`).

In [ ]:
def to_text(table) -> str:
    def cell(value):
        return "NULL" if value is None else str(value)

    lines = ["\t".join(table.column_names)]
    for row in table.to_pylist():
        lines.append("\t".join(cell(value) for value in row.values()))
    return "\n".join(lines)


text = to_text(table)
print(text)

## As a table

In [ ]:
table.to_pandas()